# 0. Imports and Configuration

In [ ]:
import polars as pl
import pandas as pd
import numpy as np

import string
from functools import lru_cache

import glob
from functools import reduce

In [ ]:
MAPPING_GENRE_FILE = "genre_mapping.txt"
IMDB_PATH = '/content/'
BUDGET_PATH = 'content/AAA Movies.csv'

# 1. Read Raw Data

## 1.1 IMDb Data

In [ ]:
dfs = {}
for path in glob.glob(IMDB_PATH + '*.tsv'):
    df = pl.read_csv(
        path,
        separator="\t",
        null_values="\\N"
    )
    name = path.split('/')[-1].replace('.tsv', '')

    print(name, df.shape, df.columns)
    dfs[name] = df

## 1.2 Budget Data

In [ ]:
budget = pd.read_csv(BUDGET_PATH, usecols=[
    'Movie Name',
    'Production Budget (USD)',
    'Release Date',
    'Genre'
]).rename(columns = {
    'Movie Name': 'title',
    'Production Budget (USD)': 'budget',
    'Release Date': 'release',
    'Genre': 'genre',
})
print(budget.shape)

# 2. Cleaning and Mapping of Existing Values

## 2.1 IMDb Data

In [ ]:
interim = dfs['title.ratings'].drop_nulls(subset=["averageRating", "numVotes"])
dfs['title.ratings'] = interim


interim = dfs['title.basics'].drop(["titleType", 'isAdult', 'endYear', 'runtimeMinutes']).drop_nulls(subset=["primaryTitle", "originalTitle"])
dfs['title.basics'] = interim


category_mapping = {
    "self": "actor", "producer": "producer", "actress": "actor",
    "editor": "crew", "production_designer": "crew", "director": "director",
    "archive_footage": "crew", "writer": "crew", "archive_sound": "crew",
    "composer": "crew", "actor": "actor", "cinematographer": "crew",
    "casting_director": "crew"
}

base = dfs['title.principals'].drop(["ordering", 'characters', 'job']).with_columns(
    category_mapped=pl.col("category").replace_strict(category_mapping, default="crew")
)

title_stats = base.group_by("tconst").agg(
    nactor=(pl.col("category_mapped") == "actor").sum(),
    ncrew=(pl.col("category_mapped") == "crew").sum(),
    director=pl.col("nconst").filter(pl.col("category") == "director").first()
)

interim = (
    base
    .filter(pl.col("category_mapped") == "actor")
    .rename({"nconst": "actor"})
    .select(["tconst", "actor"])
    .join(title_stats, on="tconst", how="left")
)

name_lookup = dfs['name.basics'].select(["nconst", "primaryName"]).unique(subset=["nconst"])

interim = (
    interim
    .join(
        name_lookup,
        left_on="director",
        right_on="nconst",
        how="left"
    )
    .with_columns(
        director=pl.coalesce("primaryName", "director")
    )
    .drop("primaryName")
    .join(
        name_lookup,
        left_on="actor",
        right_on="nconst",
        how="left"
    )
    .with_columns(
        actor=pl.coalesce("primaryName", "actor")
    )
    .drop("primaryName")
    .select(["tconst", "actor", "director", "ncrew", "nactor"])
)


dfs['title.principals'] = interim

del dfs['name.basics']
del name_lookup, base, title_stats, interim

 Converts polars dataframe into pandas

In [ ]:
lazy_dfs = [df.lazy() for df in dfs.values()]

def safe_lazy_join(left, right):
    return left.join(right, on="tconst", how="inner")

lazy_plan = reduce(safe_lazy_join, lazy_dfs)
imdb = lazy_plan.collect(engine='streaming')
imdb = imdb.to_pandas()
imdb = imdb.dropna(subset=['primaryTitle', 'startYear', 'numVotes']).query("numVotes >= 25000& startYear >= 2006")

Cleaning column types and dropping nulls

In [ ]:
imdb['primaryTitle'] = imdb['primaryTitle'].apply(lambda x: x.strip().lower().translate(str.maketrans('', '', string.punctuation)))
imdb['genres'] = imdb['genres'].apply(lambda x: x.strip().lower().translate(str.maketrans('', '', string.punctuation)) if isinstance(x, str) else 'undefined')
imdb['actor'] = imdb['actor'].apply(lambda x: x.strip().lower().translate(str.maketrans('', '', string.punctuation)) if isinstance(x, str) else 'undefined')
imdb['director'] = imdb['director'].apply(lambda x: x.strip().lower().translate(str.maketrans('', '', string.punctuation)) if isinstance(x, str) else 'undefined')

Consolidating Genre data from IMDb into genres similar to those in Budget data

In [ ]:
with open(MAPPING_GENRE_FILE, 'w') as f:
    raw_labels = f.read_text(encoding="utf-8").splitlines()

SOURCE_GENRES = [
    "action", "adult", "adventure", "animation", "biography",
    "comedy", "crime", "documentary", "drama", "family",
    "fantasy", "filmnoir", "gameshow", "history", "horror",
    "music", "musical", "mystery", "news", "realitytv",
    "romance", "scifi", "sport", "talkshow", "thriller",
    "war", "western",
]

SOURCE_GENRES = sorted(set(SOURCE_GENRES), key=len, reverse=True)

GENRE_MAPPING = {
    "action": "action",
    "adventure": "adventure",
    "musical": "musical",
    "music": "musical",
    "western": "western",
    "drama": "drama",
    "thriller": "thrillersuspense",
    "comedy": "comedy",
    "horror": "horror",
    "romance": "romantic comedy",
    "documentary": "documentary",
    "realitytv": "reality",
    "gameshow": "reality",
    "talkshow": "reality",
}

@lru_cache(maxsize=None)
def split_genres(label):
    """Split a concatenated label into recognized source genres."""
    label = str(label).strip().lower()

    if not label:
        return ()

    # Dynamic programming: find a complete segmentation.
    @lru_cache(maxsize=None)
    def solve(pos):
        if pos == len(label):
            return ()

        candidates = []

        for genre in SOURCE_GENRES:
            if label.startswith(genre, pos):
                rest = solve(pos + len(genre))

                if rest is not None:
                    candidates.append((genre,) + rest)

        if not candidates:
            return None

        # Prefer a complete segmentation with the fewest tokens.
        return min(
            candidates,
            key=lambda parts: (
                len(parts),
                tuple(-len(part) for part in parts),
            ),
        )

    result = solve(0)

    # Return the original label as an unknown token if splitting fails.
    return result if result is not None else (label,)


def map_genres(label):
    components = split_genres(label)
    mapped = set()

    for component in components:
        if component == "undefined":
            mapped.add("undefined")
        elif component in GENRE_MAPPING:
            mapped.add(GENRE_MAPPING[component])
        elif component in SOURCE_GENRES:
            mapped.add("variety")
        else:
            mapped.add("variety")

    return " ".join(sorted(mapped))


def find_unknowns(label):
    return [
        component
        for component in split_genres(label)
        if component not in SOURCE_GENRES and component != "undefined"
    ]


source_labels = sorted({
    line.strip().lower()
    for line in raw_labels
    if line.strip()
})

imdb["genres"] = imdb["genres"].map(map_genres)

In [ ]:
print(imdb.shape)
print(imdb.columns)

## Budget Data

In [ ]:
budget['title'] = budget['title'].apply(lambda x: x.strip().lower().translate(str.maketrans('', '', string.punctuation)))
budget['genre'] = budget['genre'].apply(lambda x: x.strip().lower().translate(str.maketrans('', '', string.punctuation)) if isinstance(x, str) else 'undefined')

budget.dropna(subset=['title', 'release', 'budget'], inplace=True)

budget['release'] = budget['release'].apply(lambda x: int(x.split('-')[0]) if isinstance(x, str) else x)
budget.drop_duplicates(subset=['title', 'release'], keep='first', inplace=True)

In [ ]:
budget["movie_category"] = pd.qcut(
    budget["budget"], q=4, labels=["Indie", "A", "AA", "AAA"]
)

In [ ]:
print(budget.shape)
print(budget.columns)

# 3. Merging IMDb and Budget Data

In [ ]:
found_aaa = [title
    for title in budget.title.unique()
    if title in imdb.primaryTitle.unique()
]
filtered_budget = budget[budget['title'].isin(found_aaa)]
filtered_budget.shape

In [ ]:
df = imdb.merge(
    filtered_budget,
    how="left",
    left_on=["primaryTitle", "startYear"],
    right_on=["title", "release"],
    suffixes=("", "_df2"),
    validate="many_to_one"
)

df["genres"] = df["genres"].fillna(df["genre"])

df = df.drop(columns=["title", 'genre', 'release'])

# 4. Feature Engineering

To fill in the movie categories of the other titles from IMDb, we must get an idea of the current distribution

In [ ]:
existing_budget = df[~df['budget'].isna()]

In [ ]:
existing_budget['ratio'] = existing_budget["ncrew"] / existing_budget["nactor"]
existing_budget.groupby('movie_category').agg(min=('ratio', 'min'), max=('ratio', 'max'), av=('ratio', 'mean'))

Use the maximum values as cut off thresholds for the other movie categories

In [ ]:
ratio = df["ncrew"] / df["nactor"]

conditions = [
    ratio <= 0.74,
    ratio <= 0.77,
    ratio <= 0.81,
    ratio > 0.86
]
categories = ["Indie", "A", "AA", "AAA"]

mask = df["movie_category"].isna()
new_categories = np.select(
    conditions,
    categories,
    default=None
)

df["movie_category"] = df["movie_category"].astype(object)

df.loc[mask, "movie_category"] = new_categories[mask.to_numpy()]
df["movie_category"] = df["movie_category"].astype(object)

In [ ]:
df["movie_category"].value_counts(normalize=True) * 100

Add weighted average as a way to balance votes and ratings

In [ ]:
print(
    df.groupby("movie_category")["numVotes"]
    .describe(percentiles=[0.25, 0.50, 0.60, 0.75])
    .round(2)
)

In [ ]:
for rating in ['Indie', 'A', 'AA', 'AAA']:
    temp = df[(df['movie_category'] == rating)].copy()
    temp_C = temp["averageRating"].mean()
    temp_m = temp["numVotes"].quantile(0.60)

    temp["weightedRating"] = (
        (temp["numVotes"] / (temp["numVotes"] + temp_m))
        * temp["averageRating"]
        +
        (temp_m / (temp["numVotes"] + temp_m)) * temp_C
    )
    mask = temp.index

    df.loc[mask, "weightedRating"] = temp["weightedRating"]

# Export

Note that this file will be used for the next notebook

In [ ]:
df.to_csv('complete_movie_data.csv', index=False)